In [1]:
import os, json, gc
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import cv2
from diffusers import UNet3DConditionModel, DDPMScheduler
from IPython.display import Video, display


2026-01-20 15:02:33.176266: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768921353.619157      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768921353.761939      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768921354.815935      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768921354.815973      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768921354.815975      24 computation_placer.cc:177] computation placer alr

In [2]:
DATA_ROOT = "/kaggle/input/eira-1-t2v-dataset/data/MSRVTT/MSRVTT"
VIDEO_DIR = f"{DATA_ROOT}/videos/all"
ANN_FILE  = f"{DATA_ROOT}/annotation/MSR_VTT.json"
TRAIN_TXT = f"{DATA_ROOT}/videos/train_list_new.txt"

SAVE_DIR = "/kaggle/working/t2v_quality"
os.makedirs(SAVE_DIR, exist_ok=True)

FRAME_SIZE = 96
FRAMES = 16        # ~5 seconds at 5 fps
FPS = 5

LATENT_DIM = 8
TEXT_DIM = 512
MAX_LEN = 32

VAE_EPOCHS = 10
DIFF_EPOCHS = 20
BATCH_SIZE = 1
LR = 2e-4


SAMPLE_STEPS = 50
GUIDANCE = 9.0


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [3]:
with open(ANN_FILE) as f:
    ann = json.load(f)

vid2caps = {}
for a in ann["annotations"]:
    vid2caps.setdefault(a["image_id"], []).append(a["caption"])

train_ids = [x.strip() for x in open(TRAIN_TXT)]
train_ids = train_ids[:10000]   # only 500 videos
print("Using videos:", len(train_ids))


Using videos: 7010


In [4]:
special = ["<pad>","<bos>","<eos>","<unk>"]
word2idx = {t:i for i,t in enumerate(special)}

def tok(t): return t.lower().split()

for caps in vid2caps.values():
    for c in caps:
        for w in tok(c):
            if w not in word2idx:
                word2idx[w] = len(word2idx)

def encode(text):
    ids=[word2idx["<bos>"]]
    for w in tok(text):
        ids.append(word2idx.get(w,word2idx["<unk>"]))
        if len(ids)>=MAX_LEN-1: break
    ids.append(word2idx["<eos>"])
    while len(ids)<MAX_LEN: ids.append(word2idx["<pad>"])
    return np.array(ids)


In [5]:
class MSRVTT(Dataset):
    def __init__(self, ids): self.ids = ids
    def read(self, path):
        cap = cv2.VideoCapture(path); fs=[]
        while len(fs)<FRAMES:
            r,f = cap.read()
            if not r: break
            f = cv2.resize(f,(FRAME_SIZE,FRAME_SIZE))
            f = torch.tensor(f).permute(2,0,1)/255.
            fs.append(f)
        cap.release()
        while len(fs)<FRAMES: fs.append(fs[-1])
        return torch.stack(fs)
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        vid = self.ids[i]
        v = self.read(f"{VIDEO_DIR}/{vid}.mp4")
        cap = vid2caps[vid][0]
        return {
            "video": v,
            "input_ids": torch.tensor(encode(cap)),
            "caption": cap
        }

train_loader = DataLoader(MSRVTT(train_ids), batch_size=1, shuffle=True)


In [6]:
class VideoVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv3d(3, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv3d(64, 128, 4, 2, 1), nn.ReLU(),
            nn.Conv3d(128, LATENT_DIM, 3, 1, 1)
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose3d(LATENT_DIM, 128, 3, 1, 1), nn.ReLU(),
            nn.ConvTranspose3d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose3d(64, 3, 4, 2, 1), nn.Sigmoid()
        )

    def encode(self, x): 
        return self.enc(x)

    def decode(self, z): 
        return self.dec(z)

    def forward(self, x):
        return self.decode(self.encode(x))


class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(word2idx), TEXT_DIM)
        self.pos = nn.Embedding(MAX_LEN, TEXT_DIM)
        layer = nn.TransformerEncoderLayer(TEXT_DIM,8,2048,batch_first=True)
        self.enc = nn.TransformerEncoder(layer,6)
    def forward(self, ids):
        L = ids.size(1)
        pos = torch.arange(L,device=ids.device)
        return self.enc(self.emb(ids)+self.pos(pos))

vae = VideoVAE().to(DEVICE)
text_encoder = TextEncoder().to(DEVICE)

unet = UNet3DConditionModel(
    sample_size=(FRAMES//4, FRAME_SIZE//4, FRAME_SIZE//4),
    in_channels=LATENT_DIM,
    out_channels=LATENT_DIM,
    down_block_types=(
        "CrossAttnDownBlock3D",
        "CrossAttnDownBlock3D",
        "DownBlock3D"
    ),
    up_block_types=(
        "UpBlock3D",
        "CrossAttnUpBlock3D",
        "CrossAttnUpBlock3D"
    ),
    block_out_channels=(64, 128, 256),
    cross_attention_dim=TEXT_DIM,
).to(DEVICE)

scheduler = DDPMScheduler(num_train_timesteps=500)


In [7]:
opt_vae = torch.optim.AdamW(vae.parameters(), lr=LR)
best_vae = 1e9

for e in range(1, VAE_EPOCHS+1):
    tot=0; pbar=tqdm(train_loader,desc=f"VAE {e}/{VAE_EPOCHS}")
    for b in pbar:
        v = b["video"].to(DEVICE).permute(0,2,1,3,4)
        opt_vae.zero_grad()
        r = vae(v)
        # Match time dimension
        min_t = min(r.shape[2], v.shape[2])
        r2 = r[:, :, :min_t]
        v2 = v[:, :, :min_t]
        loss = F.mse_loss(r2, v2)

        loss.backward(); opt_vae.step()
        tot+=loss.item(); pbar.set_postfix(loss=loss.item())
    avg=tot/len(train_loader)
    if avg<best_vae:
        best_vae=avg
        torch.save(vae.state_dict(), f"{SAVE_DIR}/best_vae.pt")
        print(f"⭐ New best VAE saved at epoch {e}, loss {avg:.6f}")


VAE 1/10:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best VAE saved at epoch 1, loss 0.013569


VAE 2/10:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best VAE saved at epoch 2, loss 0.006249


VAE 3/10:   0%|          | 0/7010 [00:00<?, ?it/s]

...

⭐ New best VAE saved at epoch 8, loss 0.003133


VAE 9/10:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best VAE saved at epoch 9, loss 0.002968


VAE 10/10:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best VAE saved at epoch 10, loss 0.002959


In [8]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
opt = torch.optim.AdamW(list(unet.parameters())+list(text_encoder.parameters()), lr=LR)
best_diff=1e9

for e in range(1, DIFF_EPOCHS+1):
    tot=0; pbar=tqdm(train_loader,desc=f"DIFF {e}/{DIFF_EPOCHS}")
    for b in pbar:
        v = b["video"].to(DEVICE).permute(0,2,1,3,4)
        ids = b["input_ids"].to(DEVICE)
        ids = ids.clamp(0, len(word2idx)-1)
        with torch.no_grad(): lat = vae.encode(v)
        t = torch.randint(0, 500, (1,), device=DEVICE)

        noise = torch.randn_like(lat)
        noisy = scheduler.add_noise(lat,noise,t)
        emb = text_encoder(ids)
        pred = unet(noisy,t,encoder_hidden_states=emb).sample
        loss = F.mse_loss(pred,noise)
        opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item(); pbar.set_postfix(loss=loss.item())
    avg=tot/len(train_loader)
    if avg<best_diff:
        best_diff=avg
        torch.save({"unet":unet.state_dict(),"text":text_encoder.state_dict()}, f"{SAVE_DIR}/best_diff.pt")
        print(f"⭐ New best Diffusion saved at epoch {e}, loss {avg:.6f}")


DIFF 1/20:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best Diffusion saved at epoch 1, loss 0.151870


DIFF 2/20:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best Diffusion saved at epoch 2, loss 0.121752


DIFF 3/20:   0%|          | 0/7010 [00:00<?, ?it/s]

...

DIFF 18/20:   0%|          | 0/7010 [00:00<?, ?it/s]

DIFF 19/20:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best Diffusion saved at epoch 19, loss 0.096673


DIFF 20/20:   0%|          | 0/7010 [00:00<?, ?it/s]

⭐ New best Diffusion saved at epoch 20, loss 0.094722


In [9]:
@torch.no_grad()
def generate(prompt):
    ids = torch.tensor(encode(prompt)).unsqueeze(0).to(DEVICE)
    txt = text_encoder(ids)
    null = text_encoder(torch.zeros_like(ids))
    lat = torch.randn(1,LATENT_DIM,FRAMES//2,FRAME_SIZE//2,FRAME_SIZE//2).to(DEVICE)
    scheduler.set_timesteps(SAMPLE_STEPS)
    for t in scheduler.timesteps:
        x = torch.cat([lat]*2)
        e = torch.cat([null,txt])
        p = unet(x,t,encoder_hidden_states=e).sample
        u,c = p.chunk(2)
        lat = scheduler.step(u+GUIDANCE*(c-u),t,lat).prev_sample
    return vae.decode(lat)

def save_video(t,path):
    v = t[0].permute(1,2,3,0).cpu().numpy()
    v = (v - v.min())/(v.max()-v.min()+1e-8)
    v = (v*255).astype("uint8")
    h,w=v.shape[1],v.shape[2]
    out=cv2.VideoWriter(path,cv2.VideoWriter_fourcc(*'mp4v'),FPS,(w,h))
    for f in v: out.write(f)
    out.release()
    print("Saved:", path)

video = generate("blue ocean with waves")
path = "/kaggle/working/generated_quality.mp4"
save_video(video, path)

Saved: /kaggle/working/generated_quality.mp4


In [10]:
def save_video(t, path):
    # t: [B, T, C, H, W]
    t = t.detach().cpu()

    v = t[0]                    # [T, C, H, W]
    v = v.permute(0,2,3,1)      # [T, H, W, C]
    v = (v - v.min())/(v.max()-v.min()+1e-8)
    v = (v*255).byte().numpy()

    T, H, W, C = v.shape
    out = cv2.VideoWriter(path,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          FPS,
                          (W,H))

    for f in v:
        out.write(f)
    out.release()
    print("Saved:", path)
batch = next(iter(train_loader))
orig = batch["video"][0]        # [T, C, H, W]

# Prepare for VAE: [B, C, T, H, W]
x = orig.unsqueeze(0).permute(0,2,1,3,4).to(DEVICE)

# Run VAE
recon = vae(x).detach().cpu()[0]    # [C, T, H, W]

# Back to [T, C, H, W]
recon = recon.permute(1,0,2,3)

# Save
save_video(orig.unsqueeze(0), "/kaggle/working/orig.mp4")
save_video(recon.unsqueeze(0), "/kaggle/working/recon.mp4")


Saved: /kaggle/working/orig.mp4
Saved: /kaggle/working/recon.mp4
